# image geolocation
CLIP + SigLIP 2 + DINOv2 embeddings -> geocell classifier, ensemble of 5

In [ ]:
# config. fill in the paths here
TRAIN_IMG_DIR   = "/kaggle/input/competitions/geolocation-prediction/training_dataset/noised_dataset/images"
TEST_IMG_DIR    = "/kaggle/input/datasets/<<NAME>>/final-final-test/photo_pull"
TRAIN_LABELS_CSV= "/kaggle/input/competitions/geolocation-prediction/training_dataset/noised_dataset/ground_truth_coordinates.csv"
SAMPLE_SUB_CSV  = "/kaggle/input/datasets/<<NAME>>/smaplesf/sample_submission final.csv"
COUNTRIES_GEOJSON="/kaggle/input/competitions/geolocation-prediction/country_boundaries.geojson"

CACHE_DIR   = "/kaggle/working/emb_cache"
OUT_SUB_CSV = "/kaggle/working/submission.csv"
CKPT_PATH   = "/kaggle/working/head.pt"

TOWERS = [
    dict(key="clipL.pre",       hf="openai/clip-vit-large-patch14",        kind="clip",   batch=64, dim=1024),
    dict(key="siglip2.pool",    hf="google/siglip2-so400m-patch14-384",    kind="siglip", batch=24, dim=1152),
    dict(key="dinov2.cls_mean", hf="facebook/dinov2-large",                kind="dinov2", batch=64, dim=2048),
]

_BANNED = ("streetclip", "geo-clip", "geoclip", "geolocal")
for _t in TOWERS:
    assert not any(b in _t["hf"].lower() for b in _BANNED), f"banned model: {_t['hf']}"

SEED        = 42
NUM_WORKERS = 2
MAX_IMAGES  = None

VAL_FRAC    = 0.15
GROUP_DEG   = 0.02  # ~2km groups for the val split

CELL_SWEEP  = [400, 200, 120, 80, 45, 25]
MAX_POINTS_PER_CELL = 130
MAX_SPREAD_KM = 1000.0

TAU_KM      = 60  # soft label bandwidth
SOFT_TOPK   = 20

HIDDEN      = 1024
P_DROP      = 0.35
LR          = 3e-4
WEIGHT_DECAY= 0.05
EPOCHS      = 40
BATCH       = 512
LAMBDA_CTRY = 0.3

PRED_TOPK   = 10
GATE_KM     = 500.0

PROXY_ALPHA     = 750.0
PROXY_BETA      = 1500.0
PROXY_W_CAL     = 0.5
PROXY_W_COUNTRY = 0.25
PROXY_TIGHT_KM  = 500.0

EMB_DIRS = [
    "",  # e.g. "/kaggle/input/<dataset-slug>/emb_cache"
]

# extra sources. if their embeddings are cached the images path can stay empty
SOURCES = [
    dict(tag="train", csv="/kaggle/input/datasets/<<NAME>>/csv-file-set/CSV FILESSSS/ground_truth_coordinates.csv", images=""),  # 19,002 competition train
    dict(tag="osv",   csv="/kaggle/input/datasets/<<NAME>>/csv-file-set/CSV FILESSSS/metadata.csv", images=""),  # 30,000 original extra
    dict(tag="osv2",  csv="/kaggle/input/datasets/<<NAME>>/csv-file-set/CSV FILESSSS/metadata 2.csv", images=""),  # 30,000 extra extra
]

AUTO_SCAN_KAGGLE_INPUT = True

USE_OSV    = True
OSV_WEIGHT = 0.5

#EXTRA SHI

HIDDEN       = 640
DEPTH        = 1
EXPANSION    = 2
P_DROP       = 0.45
LR           = 3e-4
WEIGHT_DECAY = 0.10
EPOCHS       = 40
BATCH        = 512
MIXUP_ALPHA  = 0.7
EMA_DECAY    = 0.999
N_ENSEMBLE   = 5

In [2]:
# imports + seed
import os, math, json, random, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(CACHE_DIR, exist_ok=True)
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

HAVE_GEO = True
try:
    import geopandas as gpd
    from shapely.geometry import Point
    from shapely.ops import unary_union, nearest_points
except Exception as e:
    HAVE_GEO = False
    print("geopandas/shapely missing -> country head + snap-to-land disabled:", e)

device: cuda | Tesla T4


In [3]:
# geo helpers. coords get averaged as 3d vectors, averaging longitude directly breaks at +-180
R_EARTH = 6371.0088

def haversine_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180.0
    lat1, lon1 = np.asarray(lat1), np.asarray(lon1)
    lat2, lon2 = np.asarray(lat2), np.asarray(lon2)
    dlat, dlon = (lat2 - lat1) * p, (lon2 - lon1) * p
    a = (np.sin(dlat/2)**2 + np.cos(lat1*p)*np.cos(lat2*p)*np.sin(dlon/2)**2)
    return 2 * R_EARTH * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def circular_mean(lats, lons, weights=None, axis=-1):
    lats = np.asarray(lats, dtype=np.float64); lons = np.asarray(lons, dtype=np.float64)
    w = np.ones_like(lats) if weights is None else np.asarray(weights, dtype=np.float64)
    la, lo = np.radians(lats), np.radians(lons)
    x = np.sum(w*np.cos(la)*np.cos(lo), axis=axis)
    y = np.sum(w*np.cos(la)*np.sin(lo), axis=axis)
    z = np.sum(w*np.sin(la), axis=axis)
    n = np.sqrt(x*x + y*y + z*z) + 1e-12
    x, y, z = x/n, y/n, z/n
    return np.degrees(np.arcsin(np.clip(z, -1, 1))), np.degrees(np.arctan2(y, x))

def pairwise_haversine(lat_a, lon_a, lat_b, lon_b, block=2048):
    lat_a, lon_a = np.asarray(lat_a), np.asarray(lon_a)
    lat_b, lon_b = np.asarray(lat_b), np.asarray(lon_b)
    out = np.empty((len(lat_a), len(lat_b)), dtype=np.float32)
    for s in range(0, len(lat_a), block):
        e = min(s + block, len(lat_a))
        out[s:e] = haversine_km(lat_a[s:e, None], lon_a[s:e, None], lat_b[None, :], lon_b[None, :])
    return out

_a, _b = circular_mean([0.0, 0.0], [179.0, -179.0])
assert abs(abs(float(_b)) - 180.0) < 1e-6
print("geo primitives ok")

geo primitives ok


In [4]:
# image loading + the 3 frozen models
from torchvision import transforms as T

def resolve_path(img_dir, img_id):
    p = os.path.join(img_dir, str(img_id))
    if os.path.splitext(p)[1]:
        return p
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG"):
        if os.path.exists(p + ext):
            return p + ext
    return p + ".jpg"

class ImgDS(Dataset):
    def __init__(self, ids, img_dir, tf):
        self.ids, self.dir, self.tf = list(ids), img_dir, tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = Image.open(resolve_path(self.dir, self.ids[i])).convert("RGB")
        return self.tf(img), i

def build_transform(proc):
    mean, std = proc.image_mean, proc.image_std
    size = proc.size
    steps = []
    if "shortest_edge" in size:
        se = size["shortest_edge"]
        crop = getattr(proc, "crop_size", None)
        crop = crop["height"] if isinstance(crop, dict) else se
        steps += [T.Resize(se, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(crop)]
    else:
        steps += [T.Resize((size["height"], size["width"]),
                           interpolation=T.InterpolationMode.BICUBIC)]
    steps += [T.ToTensor(), T.Normalize(mean, std)]
    return T.Compose(steps)

def load_tower(spec):
    from transformers import AutoImageProcessor
    kind, hf = spec["kind"], spec["hf"]
    proc = AutoImageProcessor.from_pretrained(hf)
    if kind == "clip":
        from transformers import CLIPVisionModel  # vision part only
        m = CLIPVisionModel.from_pretrained(hf)
    elif kind == "siglip":
        try:
            from transformers import SiglipVisionModel
            m = SiglipVisionModel.from_pretrained(hf)
        except Exception as e:
            print(f"  {hf} unavailable ({e}); falling back to siglip-so400m")
            from transformers import SiglipVisionModel
            hf = "google/siglip-so400m-patch14-384"
            proc = AutoImageProcessor.from_pretrained(hf)
            m = SiglipVisionModel.from_pretrained(hf)
    else:
        from transformers import AutoModel
        m = AutoModel.from_pretrained(hf)
    return m.eval().to(DEVICE), build_transform(proc)

def pool(kind, model, px):
    if kind in ("clip", "siglip"):
        return model(pixel_values=px).pooler_output
    h = model(pixel_values=px).last_hidden_state
    return torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1)  # cls + mean patch

In [ ]:
# extract embeddings for every image and cache them (skips anything already cached)
PROGRESS_SEC = 20

os.makedirs(CACHE_DIR, exist_ok=True)
import glob as _g

def _fn(tag, spec):
    return f"{tag}__{spec['key'].replace('.', '_')}"

# where to look for cached embeddings
CACHE_READ = []
for _d in EMB_DIRS:
    if not _d:
        continue
    if os.path.isdir(_d):
        CACHE_READ.append(_d)
    else:
        print(f"  ! EMB_DIRS entry is not a directory, ignoring: {_d}")
CACHE_READ.append(CACHE_DIR)
if AUTO_SCAN_KAGGLE_INPUT:
    CACHE_READ += sorted(set(os.path.dirname(p) for p in
                             _g.glob("/kaggle/input/**/*__*.npy", recursive=True)))
CACHE_READ = list(dict.fromkeys(CACHE_READ))
print("cache roots:", CACHE_READ)

def cache_paths(tag, spec):
    out = []
    for root in CACHE_READ:
        p = os.path.join(root, _fn(tag, spec) + ".npy")
        if os.path.exists(p) and p not in out:
            out.append(p)
    return out

def cache_coverage(tag, ids):
    want, out = set(str(i) for i in ids), []
    for spec in TOWERS:
        got = set()
        for p in cache_paths(tag, spec):
            sc = p[:-4] + ".ids.json"
            if os.path.exists(sc):
                got |= want & {str(x) for x in json.load(open(sc))}
            else:
                n = len(np.load(p, mmap_mode="r"))
                got |= {str(i) for i in list(ids)[:n]}
        out.append(len(got))
    return out

@torch.no_grad()
def extract(spec, tag, ids, img_dir):
    fn   = _fn(tag, spec)
    ids  = [str(i) for i in ids]
    n    = len(ids)
    pos  = {v: i for i, v in enumerate(ids)}
    out  = np.zeros((n, spec["dim"]), dtype=np.float16)
    have = np.zeros(n, dtype=bool)

    for path in cache_paths(tag, spec):
        if have.all():
            break
        A  = np.load(path, mmap_mode="r")
        sc = path[:-4] + ".ids.json"
        if os.path.exists(sc):
            cids = [str(x) for x in json.load(open(sc))]
        else:
            assert len(A) <= n, (f"{path}: {len(A)} cached rows > {n} ids in the CSV - "
                                 f"wrong CSV for this cache, or rows were removed")
            cids = ids[:len(A)]
            print(f"  ! {fn}: no sidecar, assuming prefix alignment for {len(A)} rows")
        assert len(cids) == len(A), f"{path}: {len(A)} rows vs {len(cids)} ids"
        assert A.shape[1] == spec["dim"], f"{path}: dim {A.shape[1]} != {spec['dim']}"

        pairs = [(r, pos[c]) for r, c in enumerate(cids) if c in pos and not have[pos[c]]]
        if pairs:
            src = np.array([a for a, _ in pairs]); dst = np.array([b for _, b in pairs])
            o = np.argsort(src)
            out[dst[o]] = np.asarray(A[src[o]], dtype=np.float16)
            have[dst]   = True
            print(f"  {fn:28s} +{len(pairs):>6} rows from {path}")
        del A

    todo = np.where(~have)[0]
    if len(todo) == 0:
        print(f"  {fn:28s} fully cached {out.shape} - nothing extracted")
        return out

    # anything not cached gets extracted
    assert img_dir and os.path.isdir(img_dir), (
        f"{fn}: {len(todo)}/{n} rows are in no cache, and source '{tag}' has "
        f"images={img_dir!r} which is not a directory. Either point EMB_DIRS at a "
        f"folder containing {fn}.npy, or give '{tag}' an images dir to extract from.")
    model, tf = load_tower(spec)
    dl = DataLoader(ImgDS([ids[i] for i in todo], img_dir, tf), batch_size=spec["batch"],
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    t0, last, done = time.time(), time.time(), 0
    print(f"  {spec['key']:18s} {tag}: extracting {len(todo)}/{n} missing", flush=True)
    for px, idx in dl:
        px = px.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            f = pool(spec["kind"], model, px)
        out[todo[idx.numpy()]] = f.float().cpu().numpy().astype(np.float16)
        done += len(idx)
        now = time.time()
        if now - last >= PROGRESS_SEC or done == len(todo):
            el = now - t0; rate = done / max(el, 1e-6)
            print(f"    {done:>7}/{len(todo)} ({done/len(todo):5.1%})  {rate:6.1f} img/s  "
                  f"elapsed {el/60:5.1f}m  eta {(len(todo)-done)/max(rate,1e-6)/60:5.1f}m",
                  flush=True)
            last = now
    del model; gc.collect(); torch.cuda.empty_cache()

    np.save(os.path.join(CACHE_DIR, fn + ".npy"), out)
    json.dump(ids, open(os.path.join(CACHE_DIR, fn + ".ids.json"), "w"))
    return out

_DEF = {"train": (TRAIN_LABELS_CSV, TRAIN_IMG_DIR)}

def src_paths(s):
    c, i = _DEF.get(s["tag"], ("", ""))
    return (s.get("csv") or c), (s.get("images") or i)

def read_labels(tag, path):
    L = pd.read_csv(path)
    L.columns = ["image_id", "latitude", "longitude"] + list(L.columns[3:])
    L["image_id"] = L["image_id"].astype(str)
    if MAX_IMAGES:
        L = L.iloc[:MAX_IMAGES].reset_index(drop=True)
    return L

LABELS, IMGDIR = {}, {}
for _s in SOURCES:
    _csv, _img = src_paths(_s)
    assert _csv, (f"source '{_s['tag']}': csv is empty. It carries the labels, so it "
                  f"is required whether or not the embeddings are cached.")
    LABELS[_s["tag"]] = read_labels(_s["tag"], _csv)
    IMGDIR[_s["tag"]] = _img

sub_tpl  = pd.read_csv(SAMPLE_SUB_CSV)
test_ids = [str(i) for i in sub_tpl[sub_tpl.columns[0]].tolist()]

print(f"\n{'source':8s} {'rows':>7}  {'cached (per tower)':22s} plan")
_PLAN = [(s["tag"], LABELS[s["tag"]]["image_id"].tolist(), IMGDIR[s["tag"]]) for s in SOURCES]
_PLAN.append(("test", test_ids, TEST_IMG_DIR))
for _tag, _ids, _img in _PLAN:
    _cov = cache_coverage(_tag, _ids)
    _full = all(c == len(_ids) for c in _cov)
    _plan = "all cached, no extraction" if _full else f"EXTRACT from {_img or 'NOTHING - will fail'}"
    print(f"{_tag:8s} {len(_ids):7,}  {str(_cov):22s} {_plan}")
print()

lab       = LABELS["train"]
train_ids = lab["image_id"].tolist()
print(f"train {len(train_ids)} | test {len(test_ids)}")

print("train:"); train_blocks = [(s["key"], extract(s, "train", train_ids, IMGDIR["train"])) for s in TOWERS]
print("test:");  test_blocks  = [(s["key"], extract(s, "test",  test_ids,  TEST_IMG_DIR))    for s in TOWERS]

# extra osv data
osv_lab, osv_blocks = None, None
EXTRA_SOURCES = [s for s in SOURCES if s["tag"] != "train"]
if USE_OSV and EXTRA_SOURCES:
    labs, per_tower = [], {s["key"]: [] for s in TOWERS}
    for src in EXTRA_SOURCES:
        tag = src["tag"]
        L   = LABELS[tag]
        print(f"osv [{tag}] {len(L)} rows")
        for s in TOWERS:
            per_tower[s["key"]].append(extract(s, tag, L["image_id"].tolist(), IMGDIR[tag]))
        labs.append(L)
    osv_lab    = pd.concat(labs, ignore_index=True)
    osv_blocks = [(s["key"], np.vstack(per_tower[s["key"]])) for s in TOWERS]
    del per_tower; gc.collect()
    assert len(osv_lab) == len(osv_blocks[0][1]), "osv label/embedding row mismatch"
    print(f"osv total {len(osv_lab)} rows")
elif USE_OSV:
    print("USE_OSV set but SOURCES has no non-train entries")

cache roots: ['/kaggle/working/emb_cache', '/kaggle/input/datasets/sudhanvaed25b055/new-test-dataset-embeddings/geoloc-run-artifacts-v1 (2) 2/emb_cache']

source      rows  cached (per tower)     plan
train     19,002  [19002, 19002, 19002]  all cached, no extraction
osv       30,000  [30000, 30000, 30000]  all cached, no extraction
osv2      30,000  [30000, 30000, 30000]  all cached, no extraction
test       2,448  [0, 0, 0]              EXTRACT from /kaggle/input/datasets/sudhanvaed25b055/final-final-test/photo_pull

train 19002 | test 2448
train:
  train__clipL_pre             + 19002 rows from /kaggle/input/datasets/sudhanvaed25b055/new-test-dataset-embeddings/geoloc-run-artifacts-v1 (2) 2/emb_cache/train__clipL_pre.npy
  train__clipL_pre             fully cached (19002, 1024) - nothing extracted
  train__siglip2_pool          + 19002 rows from /kaggle/input/datasets/sudhanvaed25b055/new-test-dataset-embeddings/geoloc-run-artifacts-v1 (2) 2/emb_cache/train__siglip2_pool.npy
  train

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

...

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

  dinov2.cls_mean    test: extracting 2448/2448 missing
       1856/2448 (75.8%)    90.8 img/s  elapsed   0.3m  eta   0.1m
       2448/2448 (100.0%)    91.1 img/s  elapsed   0.4m  eta   0.0m
osv [osv] 30000 rows
  osv__clipL_pre               + 30000 rows from /kaggle/input/datasets/sudhanvaed25b055/new-test-dataset-embeddings/geoloc-run-artifacts-v1 (2) 2/emb_cache/osv__clipL_pre.npy
  osv__clipL_pre               fully cached (30000, 1024) - nothing extracted
  osv__siglip2_pool            + 30000 rows from /kaggle/input/datasets/sudhanvaed25b055/new-test-dataset-embeddings/geoloc-run-artifacts-v1 (2) 2/emb_cache/osv__siglip2_pool.npy
  osv__siglip2_pool            fully cached (30000, 1152) - nothing extracted
  osv__dinov2_cls_mean         + 30000 rows from /kaggle/input/datasets/sudhanvaed25b055/new-test-dataset-embeddings/geoloc-run-artifacts-v1 (2) 2/emb_cache/osv__dinov2_cls_mean.npy
  osv__dinov2_cls_mean         fully cached (30000, 2048) - nothing extracted
osv [osv2] 30000 

In [6]:
# layernorm each model's embedding then stick them together, so one model doesn't dominate
# just because its numbers are bigger
def layernorm_concat(blocks, eps=1e-5):
    parts = []
    for _, b in blocks:
        b = b.astype(np.float32)
        parts.append((b - b.mean(1, keepdims=True)) / (b.std(1, keepdims=True) + eps))
    return np.concatenate(parts, axis=1)

print(f"{'tower':20s} {'dim':>6} {'raw L2':>9} {'post-LN L2':>11}")
VIEW_SLICES, s = [], 0
X    = layernorm_concat(train_blocks)
Xte  = layernorm_concat(test_blocks)
for (k, b) in train_blocks:
    d = b.shape[1]; VIEW_SLICES.append((k, s, s + d))
    print(f"{k:20s} {d:6d} {np.linalg.norm(b[:2000].astype(np.float32),axis=1).mean():9.2f} "
          f"{np.linalg.norm(X[:2000, s:s+d], axis=1).mean():11.2f}")
    s += d
DIM = X.shape[1]
assert Xte.shape[1] == DIM
print("concat:", X.shape, "| test:", Xte.shape)
del train_blocks, test_blocks; gc.collect()

# add the osv rows under the competition rows
LATS_ALL = lab["latitude"].to_numpy(np.float64)
LONS_ALL = ((lab["longitude"].to_numpy(np.float64) + 180) % 360) - 180
IS_OSV   = np.zeros(len(LATS_ALL), dtype=bool)

if osv_blocks is not None:
    Xosv = layernorm_concat(osv_blocks)
    assert Xosv.shape[1] == DIM, f"osv dim {Xosv.shape[1]} != {DIM}"
    assert len(Xosv) == len(osv_lab), "osv embedding/label row mismatch"
    X = np.vstack([X, Xosv]).astype(np.float32)
    LATS_ALL = np.concatenate([LATS_ALL, osv_lab["latitude"].to_numpy(np.float64)])
    _lo = ((osv_lab["longitude"].to_numpy(np.float64) + 180) % 360) - 180
    LONS_ALL = np.concatenate([LONS_ALL, _lo])
    IS_OSV   = np.concatenate([IS_OSV, np.ones(len(Xosv), dtype=bool)])
    _HAVE_X = True

assert len(LATS_ALL) == X.shape[0] == len(IS_OSV)
print(f"combined X {X.shape} | competition {(~IS_OSV).sum()} | osv {IS_OSV.sum()}")

tower                   dim    raw L2  post-LN L2
clipL.pre              1024     33.27       32.00
siglip2.pool           1152     15.09       33.94
dinov2.cls_mean        2048     51.05       45.25
concat: (19002, 4224) | test: (2448, 4224)
combined X (79002, 4224) | competition 19002 | osv 60000


In [7]:
# labels + country for each training point
lats = LATS_ALL
lons = LONS_ALL
assert len(lats) == X.shape[0], "label/embedding row mismatch"
print(f"{len(lats)} rows | |lon|>170: {(np.abs(lons) > 170).sum()}")

country_idx = np.full(len(lats), -1, dtype=np.int64)
COUNTRY_NAMES, LAND, world = [], None, None
if HAVE_GEO:
    world = gpd.read_file(COUNTRIES_GEOJSON)
    iso_col = "ISO_A2_EH" if "ISO_A2_EH" in world.columns else "ISO_A2"
    world = world[[iso_col, "geometry"]].rename(columns={iso_col: "iso"})
    pts = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(lons, lats)], crs=world.crs)
    j = gpd.sjoin(pts, world, how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    iso = j["iso"].fillna("__NA__").to_numpy()
    COUNTRY_NAMES = sorted(set(iso))
    c2i = {c: i for i, c in enumerate(COUNTRY_NAMES)}
    country_idx = np.array([c2i[c] for c in iso], dtype=np.int64)
    LAND = unary_union(world.geometry.values)
    print(f"countries {len(COUNTRY_NAMES)} | unmatched {(iso == '__NA__').sum()}")
N_COUNTRIES = max(len(COUNTRY_NAMES), 1)

79002 rows | |lon|>170: 938
countries 208 | unmatched 4390


In [8]:
# geocells: keep splitting the map into 4 until a box has <=130 points that are within 1000km of each other
def build_quadtree(lats, lons, max_points, max_spread_km=MAX_SPREAD_KM,
                   min_span_deg=0.02, max_depth=22):
    root = {"bounds": (-90., 90., -180., 180.), "idx": np.arange(len(lats)), "children": None}
    stack, leaves = [(root, 0)], []
    while stack:
        node, depth = stack.pop()
        la0, la1, lo0, lo1 = node["bounds"]
        idx = node["idx"]
        span = min(la1 - la0, lo1 - lo0)

        if len(idx) > 0:
            cla, clo = circular_mean(lats[idx], lons[idx])
            spread = float(np.max(haversine_km(lats[idx], lons[idx], cla, clo)))
        else:
            spread = 0.0

        stop = (
            (len(idx) <= max_points and spread <= max_spread_km)
            or depth >= max_depth
            or span <= min_span_deg
        )
        if stop:
            leaves.append(node); continue

        mla, mlo = 0.5*(la0+la1), 0.5*(lo0+lo1)
        la, lo = lats[idx], lons[idx]
        quads = [((la0, mla, lo0, mlo), (la <  mla) & (lo <  mlo)),
                 ((la0, mla, mlo, lo1), (la <  mla) & (lo >= mlo)),
                 ((mla, la1, lo0, mlo), (la >= mla) & (lo <  mlo)),
                 ((mla, la1, mlo, lo1), (la >= mla) & (lo >= mlo))]
        node["children"], node["split"] = [], (mla, mlo)
        for b, m in quads:
            ch = {"bounds": b, "idx": idx[m], "children": None}
            node["children"].append(ch); stack.append((ch, depth+1))
    return root, leaves

def finalize_cells(root, leaves, lats, lons):
    nonempty = [lf for lf in leaves if len(lf["idx"]) > 0]
    rlat, rlon, cnt = [], [], []
    for cid, lf in enumerate(nonempty):
        lf["cell_id"] = cid; i = lf["idx"]
        a, b = circular_mean(lats[i], lons[i])
        rlat.append(float(a)); rlon.append(float(b)); cnt.append(len(i))
    rlat, rlon, cnt = np.array(rlat), np.array(rlon), np.array(cnt)
    for lf in leaves:
        if len(lf["idx"]) == 0:
            la0, la1, lo0, lo1 = lf["bounds"]
            lf["cell_id"] = int(np.argmin(haversine_km(0.5*(la0+la1), 0.5*(lo0+lo1), rlat, rlon)))
    assign = np.empty(len(lats), dtype=np.int64)
    for lf in leaves:
        if len(lf["idx"]): assign[lf["idx"]] = lf["cell_id"]
    return assign, rlat, rlon, cnt

def assign_points(root, lats, lons):
    out = np.empty(len(lats), dtype=np.int64)
    for i in range(len(lats)):
        node = root
        while node["children"] is not None:
            mla, mlo = node["split"]
            node = node["children"][(0 if lats[i] < mla else 2) + (0 if lons[i] < mlo else 1)]
        out[i] = node["cell_id"]
    return out

In [ ]:
# how fine should the cells be? (best possible error if we always picked the right cell)
SPREAD_SWEEP = [1000, 500, 300, 200, 150, 100]
FIXED_MAX_POINTS = 200

rows = []
for sp in SPREAD_SWEEP:
    r, lv = build_quadtree(lats, lons, max_points=FIXED_MAX_POINTS, max_spread_km=sp)
    a, rla, rlo, c = finalize_cells(r, lv, lats, lons)
    fl = haversine_km(lats, lons, rla[a], rlo[a])
    rows.append(dict(max_spread_km=sp, n_cells=len(rla), median_per_cell=int(np.median(c)),
                     floor_median_km=round(float(np.median(fl)), 1),
                     floor_p90_km=round(float(np.percentile(fl, 90)), 1)))
print(pd.DataFrame(rows).to_string(index=False))

 max_spread_km  n_cells  median_per_cell  floor_median_km  floor_p90_km
          1000     1232               54             79.8         226.2
           500     1566               36             75.4         193.4
           300     2513               13             62.1         142.7
           200     3559                8             51.6         108.7
           150     5320                5             37.7          90.1
           100     8521                4             24.0          55.8


In [ ]:
# build the cells + val split. whole ~2km areas are held out so near duplicate frames don't leak.
# val is competition images only
ROOT, LEAVES = build_quadtree(lats, lons, max_points=MAX_POINTS_PER_CELL)
cell_assign, REP_LAT, REP_LON, cell_counts = finalize_cells(ROOT, LEAVES, lats, lons)
N_CELLS = len(REP_LAT)
assert (assign_points(ROOT, lats, lons) == cell_assign).all()
print(f"cells={N_CELLS} median/cell={int(np.median(cell_counts))} "
      f"min={cell_counts.min()} max={cell_counts.max()}")

def grouped_split(lats, lons, cell_assign, val_frac, grid_deg, seed):
    key = (np.round(lats/grid_deg).astype(np.int64).astype(str) + "_" +
           np.round(lons/grid_deg).astype(np.int64).astype(str))
    uniq, inv = np.unique(key, return_inverse=True)
    rng = np.random.default_rng(seed)
    remaining = np.bincount(cell_assign, minlength=cell_assign.max()+1).astype(np.int64)
    target = int(val_frac * len(lats))
    is_val = np.zeros(len(lats), dtype=bool); n_val = 0
    members = [np.where(inv == g)[0] for g in range(len(uniq))]
    for g in rng.permutation(len(uniq)):
        if n_val >= target: break
        m = members[g]
        cs, ct = np.unique(cell_assign[m], return_counts=True)
        if np.any(remaining[cs] - ct < 1): continue  # never empty a cell of train data
        remaining[cs] -= ct; is_val[m] = True; n_val += len(m)
    return is_val

comp = np.where(~IS_OSV)[0]
_v = grouped_split(lats[comp], lons[comp], cell_assign[comp], VAL_FRAC, GROUP_DEG, SEED)
IS_VAL = np.zeros(len(lats), dtype=bool)
IS_VAL[comp[_v]] = True

tr_idx, va_idx = np.where(~IS_VAL)[0], np.where(IS_VAL)[0]
assert (~IS_OSV[va_idx]).all(), "OSV leaked into validation"
print(f"train={len(tr_idx)} (osv {int(IS_OSV[tr_idx].sum())}) val={len(va_idx)} "
      f"({len(va_idx)/len(comp):.1%} of competition) | "
      f"cells in train {len(np.unique(cell_assign[tr_idx]))}/{N_CELLS}")

cells=1763 median/cell=39 min=1 max=130
train=76152 (osv 60000) val=2850 (15.0% of competition) | cells in train 1763/1763


In [11]:
# quick knn baseline to check the embeddings actually know geography
Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
K = 16
sims = Xn[va_idx] @ Xn[tr_idx].T
nn_i = np.argpartition(-sims, K, axis=1)[:, :K]
nn_s = np.take_along_axis(sims, nn_i, axis=1)
w = np.exp((nn_s - nn_s.max(1, keepdims=True)) / 0.05)
g = tr_idx[nn_i]
knn_lat, knn_lon = circular_mean(lats[g], lons[g], w, axis=1)
knn_err = haversine_km(lats[va_idx], lons[va_idx], knn_lat, knn_lon)
knn_spread = np.array([np.percentile(haversine_km(knn_lat[i], knn_lon[i], lats[g[i]], lons[g[i]]), 70)
                       for i in range(len(va_idx))])
print(f"kNN(K={K}) median {np.median(knn_err):8.1f} km | p25 {np.percentile(knn_err,25):7.1f} "
      f"| p75 {np.percentile(knn_err,75):8.1f} | spread-as-radius coverage {(knn_err<=knn_spread).mean():.3f}")
del sims; gc.collect()

kNN(K=16) median   1844.7 km | p25   784.5 | p75   4247.2 | spread-as-radius coverage 0.798


0

In [12]:
# soft labels (formula from the PIGEON paper). nearby cells get partial credit

def soft_labels_subtracted(lat, lon, cell_assign, rep_lat, rep_lon, tau_km, topk):
    D = pairwise_haversine(lat, lon, rep_lat, rep_lon)
    d_true = D[np.arange(len(lat)), cell_assign][:, None]
    k = min(topk, D.shape[1])
    idx = np.argpartition(D, k - 1, axis=1)[:, :k]
    miss = ~(idx == cell_assign[:, None]).any(axis=1)
    if miss.any():
        idx[miss, -1] = cell_assign[miss]
    d = np.take_along_axis(D, idx, axis=1).astype(np.float64)
    w_raw = np.exp(-(d - d_true) / tau_km)
    w = w_raw / w_raw.sum(axis=1, keepdims=True)
    return idx.astype(np.int64), w.astype(np.float32), w_raw

def _true_cell_mass(idx, w, cell_assign):
    hit = (idx == cell_assign[:, None])
    return np.where(hit.any(1), w[np.arange(len(w)), hit.argmax(1)], 0.0)

# try a few tau values
print(f"{'tau_km':>7} {'row sum ok':>11} {'median true-cell mass':>22} {'eff cells/label':>16}")
for _t in (20.0, 40.0, 60.0, 100.0, 150.0, 250.0):
    _i, _w, _ = soft_labels_subtracted(lats, lons, cell_assign, REP_LAT, REP_LON,
                                       _t, SOFT_TOPK)
    _eff = np.exp(-(_w * np.log(_w + 1e-12)).sum(1)).mean()
    print(f"{_t:7.0f} {str(bool(np.allclose(_w.sum(1), 1.0, atol=1e-5))):>11} "
          f"{np.median(_true_cell_mass(_i, _w, cell_assign)):22.3f} {_eff:16.2f}")

SL_IDX, SL_W, SL_RAW = soft_labels_subtracted(lats, lons, cell_assign,
                                              REP_LAT, REP_LON, TAU_KM, SOFT_TOPK)

assert np.allclose(SL_W.sum(1), 1.0, atol=1e-5), \
    f"rows do not sum to 1 (min {SL_W.sum(1).min():.3e})"
_hit = (SL_IDX == cell_assign[:, None])
assert _hit.any(1).all(), "true cell missing from the top-k"
assert np.allclose(SL_RAW[_hit], 1.0, atol=1e-9), "true-cell raw weight is not 1"
print(f"tau={TAU_KM} km | median true-cell mass "
      f"{np.median(_true_cell_mass(SL_IDX, SL_W, cell_assign)):.3f} "
      f"| effective cells/label "
      f"{np.exp(-(SL_W*np.log(SL_W+1e-12)).sum(1)).mean():.2f}")

 tau_km  row sum ok  median true-cell mass  eff cells/label
     20        True                  0.935             1.54
     40        True                  0.717             2.47
     60        True                  0.544             3.86
    100        True                  0.342             7.25
    150        True                  0.228            10.82
    250        True                  0.142            14.91

tau=60 km | median true-cell mass 0.544 | effective cells/label 3.86


In [13]:
# model
class ResBlock(nn.Module):
    def __init__(self, dim, expansion=EXPANSION, p=P_DROP):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim * expansion)
        self.fc2 = nn.Linear(dim * expansion, dim)
        self.drop = nn.Dropout(p)

    def forward(self, x):
        return x + self.drop(self.fc2(F.gelu(self.fc1(self.norm(x)))))

class GeoHead(nn.Module):
    def __init__(self, view_slices, dim, n_cells, n_countries,
                 hidden=HIDDEN, depth=DEPTH, p=P_DROP):
        super().__init__()
        vid = torch.zeros(dim, dtype=torch.long)
        for k, (_, s, e) in enumerate(view_slices):
            vid[s:e] = k
        self.register_buffer("view_id", vid)
        self.gates = nn.Parameter(torch.ones(len(view_slices)))

        self.stem = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(p))
        self.blocks = nn.ModuleList([ResBlock(hidden, EXPANSION, p) for _ in range(depth)])
        self.out_norm = nn.LayerNorm(hidden)
        self.cell_head = nn.Linear(hidden, n_cells)
        self.ctry_head = nn.Linear(hidden, n_countries)

    def forward(self, x):
        h = self.stem(x * self.gates[self.view_id])
        for blk in self.blocks:
            h = blk(h)
        h = self.out_norm(h)
        return self.cell_head(h), self.ctry_head(h)

def soft_ce(logits, idx, w, sw=None):
    per = -(torch.gather(F.log_softmax(logits, 1), 1, idx) * w).sum(1)
    return per.mean() if sw is None else (per * sw).sum() / (sw.sum() + 1e-8)

class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items() if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)

    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items():
            sd[k].copy_(v.to(sd[k].dtype))

Xt   = torch.from_numpy(X).to(DEVICE)
SIt  = torch.from_numpy(SL_IDX).to(DEVICE)
SWt  = torch.from_numpy(SL_W).to(DEVICE)
CTt  = torch.from_numpy(country_idx).to(DEVICE)
HARD = torch.from_numpy(cell_assign).to(DEVICE)
TR   = torch.from_numpy(tr_idx).to(DEVICE)
VA   = torch.from_numpy(va_idx).to(DEVICE)
SWGT = torch.from_numpy(np.where(IS_OSV, OSV_WEIGHT, 1.0).astype(np.float32)).to(DEVICE)

_probe = GeoHead(VIEW_SLICES, DIM, N_CELLS, N_COUNTRIES)
_np = sum(p.numel() for p in _probe.parameters())
print(f"on {DEVICE}: X {tuple(Xt.shape)}, {Xt.element_size()*Xt.nelement()/1e6:.0f} MB")
print(f"head: {_np/1e6:.1f}M params | hidden {HIDDEN} depth {DEPTH} drop {P_DROP}")
print(f"{len(tr_idx)} train rows over {N_CELLS} cells = "
      f"{len(tr_idx)/N_CELLS:.0f} rows/cell | {_np/len(tr_idx):.0f} params per training row")
print(f"ensemble of {N_ENSEMBLE} | mixup alpha {MIXUP_ALPHA} | ema {EMA_DECAY}")
del _probe

on cuda: X (79002, 4224), 1335 MB
head: 5.6M params | hidden 640 depth 1 drop 0.45
76152 train rows over 1763 cells = 43 rows/cell | 74 params per training row
ensemble of 5 | mixup alpha 0.7 | ema 0.999


In [ ]:
# train 5 models with different seeds and average them

use_ctry = HAVE_GEO and bool((country_idx >= 0).any())

@torch.no_grad()
def _logits_for(model, index_tensor, chunk=4096):
    model.eval()
    return torch.cat([model(Xt[index_tensor[s:s+chunk]])[0].float()
                      for s in range(0, len(index_tensor), chunk)])

def train_one(seed):
    set_seed(seed)
    model = GeoHead(VIEW_SLICES, DIM, N_CELLS, N_COUNTRIES).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = EPOCHS * max(1, len(tr_idx) // BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps,
                                                pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    ema = EMA(model) if EMA_DECAY else None
    g_cpu = torch.Generator(); g_cpu.manual_seed(seed)

    for ep in range(EPOCHS):
        model.train()
        perm = TR[torch.randperm(len(TR), generator=g_cpu).to(DEVICE)]
        tot, nb = 0.0, 0
        for s in range(0, len(perm) - BATCH + 1, BATCH):
            b = perm[s:s+BATCH]
            opt.zero_grad(set_to_none=True)

            # mixup
            lam, perm_b = 1.0, None
            if MIXUP_ALPHA > 0:
                lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                lam = max(lam, 1.0 - lam)
                perm_b = b[torch.randperm(len(b), device=DEVICE)]

            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                xb = Xt[b] if perm_b is None else lam * Xt[b] + (1 - lam) * Xt[perm_b]
                cl, co = model(xb)
                loss = soft_ce(cl, SIt[b], SWt[b], SWGT[b])
                if perm_b is not None:
                    loss = lam * loss + (1 - lam) * soft_ce(cl, SIt[perm_b], SWt[perm_b],
                                                            SWGT[perm_b])
                if use_ctry:
                    ce = F.cross_entropy(co, CTt[b], ignore_index=-1, reduction="none")
                    ctry = (ce * SWGT[b]).sum() / (SWGT[b].sum() + 1e-8)
                    if perm_b is not None:
                        ce2 = F.cross_entropy(co, CTt[perm_b], ignore_index=-1,
                                              reduction="none")
                        ctry = lam * ctry + (1 - lam) * (
                            (ce2 * SWGT[perm_b]).sum() / (SWGT[perm_b].sum() + 1e-8))
                    loss = loss + LAMBDA_CTRY * ctry

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update(model)
            tot += loss.item(); nb += 1

        if ep % 10 == 0 or ep == EPOCHS - 1:
            vl = _logits_for(model, VA)
            am = vl.argmax(1).cpu().numpy()
            acc = float((vl.argmax(1) == HARD[VA]).float().mean())
            med = np.median(haversine_km(lats[va_idx], lons[va_idx], REP_LAT[am], REP_LON[am]))
            print(f"    ep {ep:3d} loss {tot/max(nb,1):.4f} | val top1 {acc:.3f} | "
                  f"argmax-cell median {med:8.1f} km")

    if ema is not None:
        def _med(lg):
            am = lg.argmax(1).cpu().numpy()
            return float(np.median(haversine_km(lats[va_idx], lons[va_idx],
                                                REP_LAT[am], REP_LON[am])))
        raw_med = _med(_logits_for(model, VA))
        ema.copy_to(model)
        ema_med = _med(_logits_for(model, VA))
        note = "kept ema" if ema_med <= raw_med else "ema worse; kept anyway, it decorrelates members"
        print(f"    ema {ema_med:.1f} km vs raw {raw_med:.1f} km  ({note})")

    return model

MODELS = []
for _e in range(N_ENSEMBLE):
    print(f"  member {_e+1}/{N_ENSEMBLE} (seed {SEED + 1000*_e})")
    MODELS.append(train_one(SEED + 1000 * _e))
    print(f"    gates {MODELS[-1].gates.detach().cpu().numpy().round(3)}")

model = MODELS[0]

@torch.no_grad()
def ensemble_logprobs(index_tensor=None, X_rows=None, chunk=4096):
    outs = []
    for s in range(0, (len(index_tensor) if X_rows is None else len(X_rows)), chunk):
        xb = (Xt[index_tensor[s:s+chunk]] if X_rows is None else X_rows[s:s+chunk])
        p = None
        for m in MODELS:
            m.eval()
            q = F.softmax(m(xb)[0].float(), 1)
            p = q if p is None else p + q
        outs.append(torch.log((p / len(MODELS)).clamp_min(1e-12)))
    return torch.cat(outs)

def val_logits():
    return ensemble_logprobs(index_tensor=VA)

_vl = val_logits()
_am = _vl.argmax(1).cpu().numpy()
print(f"\nensemble of {len(MODELS)}: val top1 "
      f"{float((_vl.argmax(1) == HARD[VA]).float().mean()):.3f} | argmax-cell median "
      f"{np.median(haversine_km(lats[va_idx], lons[va_idx], REP_LAT[_am], REP_LON[_am])):.1f} km")

torch.save({"models": [m.state_dict() for m in MODELS],
            "rep_lat": REP_LAT, "rep_lon": REP_LON, "view_slices": VIEW_SLICES,
            "n_cells": N_CELLS, "n_countries": N_COUNTRIES, "dim": DIM,
            "hidden": HIDDEN, "depth": DEPTH, "n_ensemble": len(MODELS),
            "use_osv": bool(USE_OSV), "osv_weight": float(OSV_WEIGHT)}, CKPT_PATH)
print("saved", CKPT_PATH)

  member 1/5 (seed 42)
    ep   0 loss 8.6888 | val top1 0.021 | argmax-cell median   6409.3 km
    ep  10 loss 4.2084 | val top1 0.183 | argmax-cell median    476.9 km
    ep  20 loss 3.6222 | val top1 0.196 | argmax-cell median    429.6 km
    ep  30 loss 3.3726 | val top1 0.206 | argmax-cell median    410.3 km
    ep  39 loss 3.2132 | val top1 0.209 | argmax-cell median    401.3 km
    ema 403.0 km vs raw 401.3 km  (ema worse; kept anyway, it decorrelates members)
    gates [0.784 0.817 0.738]
  member 2/5 (seed 1042)
    ep   0 loss 8.7095 | val top1 0.013 | argmax-cell median   6907.4 km
    ep  10 loss 4.2278 | val top1 0.184 | argmax-cell median    461.1 km
    ep  20 loss 3.6725 | val top1 0.202 | argmax-cell median    420.7 km
    ep  30 loss 3.4332 | val top1 0.196 | argmax-cell median    413.6 km
    ep  39 loss 3.1888 | val top1 0.199 | argmax-cell median    404.0 km
    ema 403.2 km vs raw 404.0 km  (kept ema)
    gates [0.783 0.818 0.733]
  member 3/5 (seed 2042)
    ep  

In [15]:
# temperature scaling, the model is overconfident otherwise
VL, yv = val_logits().detach(), HARD[VA]
logT = torch.zeros(1, device=DEVICE, requires_grad=True)
optT = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)
def closure():
    optT.zero_grad(); l = F.cross_entropy(VL / logT.exp(), yv); l.backward(); return l
optT.step(closure)
TEMP = float(logT.exp().item())
print(f"T = {TEMP:.3f} (>1 means the head was overconfident) | NLL "
      f"{F.cross_entropy(VL, yv).item():.4f} -> {F.cross_entropy(VL/TEMP, yv).item():.4f}")
val_probs = F.softmax(VL / TEMP, 1).cpu().numpy().astype(np.float64)
val_probs /= val_probs.sum(1, keepdims=True)

T = 1.177 (>1 means the head was overconfident) | NLL 3.3364 -> 3.2863


In [ ]:
# the scoring metric (worked out from probe submissions) + the old baseline methods
#   score = A*exp(-d/alpha) + B*(+-1)*exp(-r/beta) + C*[country right]
#   leaderboard = median over images

class Metric:
    __slots__ = ("A", "alpha_km", "B", "beta_km", "C")

    def __init__(self, A=100.0, alpha_km=1493.0, B=14.9996, beta_km=500.24, C=0.8):
        self.A, self.alpha_km = A, alpha_km
        self.B, self.beta_km = B, beta_km
        self.C = C

    def per_image(self, d_km, r_km, country_ok):
        d = np.asarray(d_km, np.float64)
        r = np.asarray(r_km, np.float64)
        return (self.A * np.exp(-d / self.alpha_km)
                + self.B * np.where(d <= r, 1.0, -1.0) * np.exp(-r / self.beta_km)
                + self.C * np.asarray(country_ok, np.float64))

    def score(self, d_km, r_km, country_ok):
        return float(np.median(self.per_image(d_km, r_km, country_ok)))

    def __repr__(self):
        return (f"Metric(A={self.A}, alpha={self.alpha_km}km, B={self.B:.4f}, "
                f"beta={self.beta_km:.2f}km, C={self.C})")

METRIC = Metric()
print(METRIC)

def predict_point(probs, rlat, rlon, topk=PRED_TOPK, gate_km=GATE_KM):
    k = min(topk, probs.shape[1])
    top = np.argpartition(-probs, k - 1, axis=1)[:, :k]
    tp = np.take_along_axis(probs, top, axis=1)
    o = np.argsort(-tp, axis=1)
    top, tp = np.take_along_axis(top, o, axis=1), np.take_along_axis(tp, o, axis=1)
    clat, clon = rlat[top], rlon[top]
    d = haversine_km(rlat[top[:, 0]][:, None], rlon[top[:, 0]][:, None], clat, clon)
    keep = d <= gate_km
    keep[:, 0] = True
    return circular_mean(clat, clon, tp * keep, axis=1)

def predict_radius(probs, plat, plon, rlat, rlon, q):
    D = pairwise_haversine(plat, plon, rlat, rlon)
    o = np.argsort(D, axis=1)
    Ds, Ps = np.take_along_axis(D, o, axis=1), np.take_along_axis(probs, o, axis=1)
    cum = np.cumsum(Ps, axis=1)
    hit = cum >= q
    first = np.where(hit.any(1), hit.argmax(1), Ps.shape[1] - 1)
    return Ds[np.arange(len(plat)), first]

def snap_to_land(plat, plon):
    if not HAVE_GEO or LAND is None:
        return plat, plon
    la, lo = np.asarray(plat).copy(), np.asarray(plon).copy()
    for i in range(len(la)):
        p = Point(lo[i], la[i])
        if not LAND.contains(p):
            q = nearest_points(LAND, p)[0]
            la[i], lo[i] = q.y, q.x
    return la, lo

Metric(A=100.0, alpha=1493.0km, B=14.9996, beta=500.24km, C=0.8)


In [17]:
# turn cell probabilities into a lat/lon + radius

import math

def _as_t(x, dev, dt=torch.float32):
    return torch.as_tensor(np.asarray(x), device=dev, dtype=dt)

def torch_haversine_km(lat1, lon1, lat2, lon2):
    p = math.pi / 180.0
    dlat, dlon = (lat2 - lat1) * p, (lon2 - lon1) * p
    a = (torch.sin(dlat / 2) ** 2
         + torch.cos(lat1 * p) * torch.cos(lat2 * p) * torch.sin(dlon / 2) ** 2)
    return 2 * R_EARTH * torch.asin(torch.sqrt(a.clamp(0.0, 1.0)))

def coverage_radius_for_score(median_score, metric=None):
    metric = metric or METRIC
    m = float(np.clip(median_score, 1e-6, metric.A - 1e-6))
    return float(metric.alpha_km * math.log(metric.A / m))

@torch.no_grad()
def decide(probs, cell_of_ref, ref_lat, ref_lon, ref_country=None, metric=None,
           topm=1024, r_min_km=0.0, utility="coverage", cov_R_km=1035.0,
           refine_km=None, r_bail_km=20100.0, mem_budget=6e7, verbose=False):
    metric = metric or METRIC
    probs = np.asarray(probs, np.float64)
    N, n_cells = probs.shape
    n_ref = len(ref_lat)
    topm = int(min(topm, n_ref))
    if refine_km is None:
        refine_km = cov_R_km if utility == "coverage" else metric.alpha_km / 4.0

    cop = _as_t(cell_of_ref, DEVICE, torch.long)
    inv_count = 1.0 / torch.bincount(cop, minlength=n_cells).clamp_min(1).to(torch.float32)
    rla, rlo = _as_t(ref_lat, DEVICE), _as_t(ref_lon, DEVICE)
    rct = _as_t(ref_country, DEVICE, torch.long) if ref_country is not None else None
    n_ctry = int(rct.max().item()) + 1 if rct is not None else 0

    chunk = max(1, int(mem_budget // (topm * topm)))
    if verbose:
        print(f"decide: N={N} n_ref={n_ref} topm={topm} chunk={chunk} "
              f"util={utility} R={cov_R_km:.0f}")

    o_lat = np.empty(N); o_lon = np.empty(N); o_r = np.empty(N)
    o_u = np.empty(N); o_cov = np.empty(N); o_bail = np.zeros(N, bool)

    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        P = _as_t(probs[s:e], DEVICE)
        W = P[:, cop] * inv_count[cop][None, :]  # [c, n_ref]
        W = W / W.sum(1, keepdim=True).clamp_min(1e-30)

        wv, wi = torch.topk(W, topm, dim=1)
        wn = wv / wv.sum(1, keepdim=True).clamp_min(1e-30)
        cla, clo = rla[wi], rlo[wi]

        D = torch_haversine_km(cla[:, :, None], clo[:, :, None],
                               cla[:, None, :], clo[:, None, :])  # [c, cand, supp]
        U = (D <= cov_R_km).to(D.dtype) if utility == "coverage" \
            else torch.exp(-D / metric.alpha_km)
        val = metric.A * torch.bmm(U, wn[:, :, None]).squeeze(-1)
        del U, D
        # small tie break so the pick doesn't wander
        val = val + 1e-4 * metric.A * wn

        if rct is not None and metric.C:
            cc = rct[wi]
            mass = torch.zeros(e - s, n_ctry, device=DEVICE)
            mass.scatter_add_(1, cc, wn)
            val = val + metric.C * torch.gather(mass, 1, cc)
            del mass, cc

        rows = torch.arange(e - s, device=DEVICE)
        best = val.argmax(1)
        plat, plon = cla[rows, best], clo[rows, best]
        o_u[s:e] = val[rows, best].float().cpu().numpy()
        del val, wv, wn, wi, cla, clo

        # move to the middle of the winning cluster
        Dq = torch_haversine_km(plat[:, None], plon[:, None], rla[None, :], rlo[None, :])
        if refine_km and refine_km > 0:
            m = (Dq <= refine_km).to(W.dtype) * W
            la_r, lo_r = torch.deg2rad(rla), torch.deg2rad(rlo)
            cx = (m * torch.cos(la_r) * torch.cos(lo_r)).sum(1)
            cy = (m * torch.cos(la_r) * torch.sin(lo_r)).sum(1)
            cz = (m * torch.sin(la_r)).sum(1)
            nrm = torch.sqrt(cx * cx + cy * cy + cz * cz)
            good = nrm > 1e-12
            plat = torch.where(good, torch.rad2deg(
                torch.asin((cz / nrm.clamp_min(1e-12)).clamp(-1, 1))), plat)
            plon = torch.where(good, torch.rad2deg(torch.atan2(cy, cx)), plon)
            Dq = torch_haversine_km(plat[:, None], plon[:, None],
                                    rla[None, :], rlo[None, :])
            del m, la_r, lo_r, cx, cy, cz, nrm, good

        # best radius for the score formula
        Ds, order = torch.sort(Dq, dim=1)
        Fc = torch.gather(W, 1, order).cumsum(1).clamp(0.0, 1.0)
        # option to give up with a huge radius
        Ds = torch.cat([Ds, torch.full((e - s, 1), r_bail_km, device=DEVICE)], 1)
        Fc = torch.cat([Fc, torch.ones(e - s, 1, device=DEVICE)], 1)
        obj = torch.exp(-Ds / metric.beta_km) * (2.0 * Fc - 1.0)
        obj = obj.masked_fill(Ds < r_min_km, -1e30)
        b = obj.argmax(1)

        o_r[s:e] = Ds[rows, b].float().cpu().numpy()
        o_cov[s:e] = Fc[rows, b].float().cpu().numpy()
        o_bail[s:e] = (Ds[rows, b] >= 10_000.0).cpu().numpy()
        o_lat[s:e] = plat.double().cpu().numpy()
        o_lon[s:e] = plon.double().cpu().numpy()
        del Dq, Ds, Fc, obj, W, P

    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return dict(lat=o_lat, lon=((o_lon + 180) % 360) - 180, radius_km=o_r,
                util=o_u, coverage_at_r=o_cov, bailed=o_bail)

def evaluate(name, plat, plon, radius_km, true_lat, true_lon, country_ok=None,
             metric=None):
    metric = metric or METRIC
    d = haversine_km(true_lat, true_lon, plat, plon)
    r = np.asarray(radius_km, np.float64)
    ok = np.zeros(len(d)) if country_ok is None else np.asarray(country_ok, float)
    return dict(policy=name,
                score=metric.score(d, r, ok),
                median_err_km=float(np.median(d)),
                median_r_km=float(np.median(r)),
                coverage=float((d <= r).mean()),
                median_cal=float(np.median(metric.B * np.where(d <= r, 1, -1)
                                           * np.exp(-r / metric.beta_km))))

def print_table(rows):
    hdr = f"{'policy':>26} {'score':>8} {'med err':>9} {'med r':>8} {'cov':>6} {'cal':>8}"
    print(hdr); print("-" * len(hdr))
    for r in sorted(rows, key=lambda x: -x["score"]):
        print(f"{r['policy']:>26} {r['score']:8.3f} {r['median_err_km']:9.1f} "
              f"{r['median_r_km']:8.0f} {r['coverage']:6.3f} {r['median_cal']:+8.3f}")

def decide_iterated(probs, cell_of_ref, ref_lat, ref_lon, true_lat, true_lon,
                    ref_country=None, metric=None, n_iter=5, init_R_km=1035.0,
                    country_ok_fn=None, verbose=True, **kw):
    metric = metric or METRIC
    R = float(init_R_km)
    hist, out = [], None
    for it in range(n_iter):
        out = decide(probs, cell_of_ref, ref_lat, ref_lon, ref_country=ref_country,
                     metric=metric, utility="coverage", cov_R_km=R, **kw)
        ok = None if country_ok_fn is None else country_ok_fn(out["lat"], out["lon"])
        row = evaluate(f"likelihood R={R:.0f}", out["lat"], out["lon"],
                       out["radius_km"], true_lat, true_lon, ok, metric)
        hist.append(row)
        R_next = coverage_radius_for_score(row["score"], metric)
        if verbose:
            print(f"  iter {it}: R={R:7.0f} km -> score {row['score']:7.3f}, "
                  f"median err {row['median_err_km']:7.1f} km -> R'={R_next:.0f}")
        if abs(R_next - R) < 0.02 * R:
            R = R_next
            break
        R = 0.5 * R + 0.5 * R_next
    return out, R, hist

print("likelihood estimator ready")

likelihood estimator ready


In [18]:
# validation: compare radius options
vlat, vlon = lats[va_idx], lons[va_idx]

def country_match(plat, plon, idx):
    if not HAVE_GEO:
        return np.zeros(len(plat), dtype=bool)
    pg = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(plon, plat)],
                          crs=world.crs)
    j = gpd.sjoin(pg, world, how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    got = j["iso"].fillna("__NA__").to_numpy()
    return np.array([bool(COUNTRY_NAMES[country_idx[i]] == got[n])
                     for n, i in enumerate(idx)])

REF = tr_idx
print(f"reference set: {len(REF)} training points "
      f"({int(IS_OSV[REF].sum())} osv) over {N_CELLS} cells\n")

val_out, R_STAR, _hist = decide_iterated(
    val_probs, cell_assign[REF], lats[REF], lons[REF], vlat, vlon,
    ref_country=country_idx[REF], metric=METRIC, topm=1024,
    country_ok_fn=lambda la, lo: country_match(la, lo, va_idx))
print(f"\nconverged R* = {R_STAR:.0f} km\n")

rows = []
_la, _lo = snap_to_land(val_out["lat"], val_out["lon"])
rows.append(evaluate("likelihood + policy r", _la, _lo, val_out["radius_km"],
                     vlat, vlon, country_match(_la, _lo, va_idx)))
for _rc in (500, 1000, 2000, 5000):
    rows.append(evaluate(f"likelihood + flat r={_rc}", _la, _lo,
                         np.full(len(_la), float(_rc)), vlat, vlon,
                         country_match(_la, _lo, va_idx)))

# old methods for comparison
_ea = decide(val_probs, cell_assign[REF], lats[REF], lons[REF],
             ref_country=country_idx[REF], metric=METRIC, topm=1024, utility="exp")
_ela, _elo = snap_to_land(_ea["lat"], _ea["lon"])
rows.append(evaluate("expected-score (mean rule)", _ela, _elo, _ea["radius_km"],
                     vlat, vlon, country_match(_ela, _elo, va_idx)))

for _tk, _gk in ((1, 500.0), (10, 500.0), (10, 1000.0)):
    _pa, _po = predict_point(val_probs, REP_LAT, REP_LON, _tk, _gk)
    _pa, _po = snap_to_land(_pa, _po)
    _ok = country_match(_pa, _po, va_idx)
    for _rc in (2000,):
        rows.append(evaluate(f"gated mean k={_tk} g={_gk:.0f} r={_rc}", _pa, _po,
                             np.full(len(_pa), float(_rc)), vlat, vlon, _ok))

print_table(rows)

BEST = max(rows, key=lambda r: r["score"])
RADIUS_MODE = BEST["policy"]
print(f"\nchosen: {RADIUS_MODE}  (val median score {BEST['score']:.3f})")
print(f"bailed on {val_out['bailed'].mean():.1%} of val images | "
      f"median coverage at chosen r {np.median(val_out['coverage_at_r']):.3f}")

reference set: 76152 training points (60000 osv) over 1763 cells

  iter 0: R=   1035 km -> score  82.553, median err   366.5 km -> R'=286
  iter 1: R=    661 km -> score  83.506, median err   348.1 km -> R'=269
  iter 2: R=    465 km -> score  83.327, median err   349.5 km -> R'=272
  iter 3: R=    369 km -> score  83.207, median err   350.4 km -> R'=274
  iter 4: R=    322 km -> score  83.045, median err   355.9 km -> R'=277

converged R* = 299 km

                    policy    score   med err    med r    cov      cal
----------------------------------------------------------------------
   likelihood + flat r=500   84.904     356.2      500  0.604   +5.521
     likelihood + policy r   82.905     356.2      678  0.747   +2.521
expected-score (mean rule)   82.844     356.8      666  0.748   +2.775
  likelihood + flat r=1000   81.415     356.2     1000  0.756   +2.032
gated mean k=10 g=1000 r=2000   79.856     349.2     2000  0.841   +0.275
  likelihood + flat r=2000   79.658     356.2

In [19]:
# check coverage of the chosen radius
_pl, _po = snap_to_land(val_out["lat"], val_out["lon"])
err = haversine_km(vlat, vlon, _pl, _po)

if RADIUS_MODE.startswith("likelihood + flat"):
    r = np.full(len(err), float(RADIUS_MODE.split("=")[1]))
else:
    r = val_out["radius_km"]

ok = country_match(_pl, _po, va_idx)
print(f"policy: {RADIUS_MODE} | median score {METRIC.score(err, r, ok):.3f} "
      f"| median err {np.median(err):.1f} km | country acc {ok.mean():.3f}")

qs = np.unique(np.quantile(r, [0, .2, .4, .6, .8, 1.0]))
print(f"\n{'radius band (km)':>22} {'n':>6} {'coverage':>9} {'median err':>11} {'median cal':>11}")
for i in range(len(qs) - 1):
    m = (r >= qs[i]) & (r <= qs[i + 1])
    if m.sum():
        cal = np.median(METRIC.B * np.where(err[m] <= r[m], 1, -1)
                        * np.exp(-r[m] / METRIC.beta_km))
        print(f"{qs[i]:10.0f}-{qs[i+1]:<11.0f} {int(m.sum()):6d} "
              f"{(err[m] <= r[m]).mean():9.3f} {np.median(err[m]):11.1f} {cal:11.3f}")

print(f"\nfraction sent to a give-up radius: {float((r >= 10000).mean()):.2f}")
print(f"Overall coverage {(err <= r).mean():.3f} | median calibration "
      f"{np.median(METRIC.B*np.where(err <= r, 1, -1)*np.exp(-r/METRIC.beta_km)):+.3f}")

policy: likelihood + flat r=500 | median score 84.904 | median err 356.2 km | country acc 0.665

      radius band (km)      n  coverage  median err  median cal

fraction sent to a give-up radius: 0.00
Overall coverage 0.604 | median calibration +5.521


In [20]:
# predict on test + write the submission. also writes flat radius files to probe the leaderboard
Xte_t = torch.from_numpy(Xte).to(DEVICE)
test_logits = ensemble_logprobs(X_rows=Xte_t)
test_probs = F.softmax(test_logits / TEMP, 1).cpu().numpy().astype(np.float64)

test_probs /= test_probs.sum(1, keepdims=True)

test_out = decide(test_probs, cell_assign[REF], lats[REF], lons[REF],
                  ref_country=country_idx[REF], metric=METRIC, topm=1024,
                  utility="coverage", cov_R_km=R_STAR, verbose=True)

tlat, tlon = snap_to_land(test_out["lat"], test_out["lon"])
tlon = ((tlon + 180) % 360) - 180

if RADIUS_MODE.startswith("likelihood + flat"):
    trad = np.full(len(tlat), float(RADIUS_MODE.split("=")[1]))
else:
    trad = test_out["radius_km"]

sub = sub_tpl.copy()
id_col, lat_col, lon_col, rad_col = sub.columns[:4]
sub[lat_col], sub[lon_col], sub[rad_col] = tlat, tlon, trad

assert len(sub) == len(test_ids)
assert sub[[lat_col, lon_col, rad_col]].notna().all().all()
assert np.isfinite(sub[[lat_col, lon_col, rad_col]].to_numpy()).all()
assert sub[lat_col].between(-90, 90).all() and sub[lon_col].between(-180, 180).all()
assert (sub[rad_col] > 0).all()
assert sub[id_col].is_unique
sub.to_csv(OUT_SUB_CSV, index=False)
print(sub.head())
print(f"wrote {OUT_SUB_CSV} | {len(sub)} rows | policy: {RADIUS_MODE} | R*={R_STAR:.0f} km")
print(f"radius median {np.median(trad):.0f} km | "
      f"bailed {test_out['bailed'].mean():.2%} | "
      f"median coverage at chosen r {np.median(test_out['coverage_at_r']):.3f}")

# flat radius probe files
os.makedirs(PROBE_DIR := "/kaggle/working/probes", exist_ok=True)
for R in (500, 1000, 2000, 5000):
    p = sub.copy(); p[rad_col] = float(R)
    p.to_csv(os.path.join(PROBE_DIR, f"probe_r{R}.csv"), index=False)
    print("wrote", f"probe_r{R}.csv")

decide: N=2448 n_ref=76152 topm=1024 chunk=57 util=coverage R=299
               image_id   pred_lat   pred_long  pred_radius_km
0  a88766ed092cd328.jpg  55.055302   21.573009           500.0
1  1d1b147ac111029f.jpg   1.343878  103.740501           500.0
2  8703a028b9392b48.jpg  55.101604   20.706317           500.0
3  4366f03b294ff3d7.jpg  14.414130  -60.845204           500.0
4  251dd7fa406aeaa1.jpg  58.510972   21.832367           500.0
wrote /kaggle/working/submission.csv | 2448 rows | policy: likelihood + flat r=500 | R*=299 km
radius median 500 km | bailed 1.63% | median coverage at chosen r 0.755
wrote probe_r500.csv
wrote probe_r1000.csv
wrote probe_r2000.csv
wrote probe_r5000.csv


run all. set `MAX_IMAGES = 2000` in the config cell for a quick test run first.